In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd "/content/drive/MyDrive/Courses/AI/masked_attention/llama_like"

/content/drive/MyDrive/Courses/AI/masked_attention/llama_like


In [3]:
import os
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from src.data import get_corpus
from src.tokenizer import BPETokenizer
from src.model import MiniLLaMA, ModelConfig

In [4]:
# ---------------------------------------------------------------------------
# Hyperparameters — easy to tweak
# ---------------------------------------------------------------------------
class TrainConfig:
    # Tokenizer
    bpe_merges: int   = 200

    # Model
    d_model: int      = 128
    n_heads: int      = 4
    n_kv_heads: int   = 2
    d_ff: int         = 256
    max_seq_len: int  = 64
    dropout: float    = 0.1

    # Training
    batch_size: int   = 8
    epochs: int       = 200
    lr: float         = 3e-3
    weight_decay: float = 0.01

    # Checkpointing
    save_dir: str     = "checkpoints"
    log_every: int    = 10   # print loss every N epochs


In [5]:
# ---------------------------------------------------------------------------
# Dataset: sliding window over the token sequence
# ---------------------------------------------------------------------------
class TextDataset(Dataset):
    """
    Splits a token sequence into overlapping windows of length `seq_len`.

    input  = tokens[i   : i+seq_len]
    target = tokens[i+1 : i+seq_len+1]   (next-token prediction)
    """

    def __init__(self, token_ids: list[int], seq_len: int):
        self.seq_len = seq_len
        self.data    = torch.tensor(token_ids, dtype=torch.long)
        # number of complete windows
        self.n       = max(0, len(self.data) - seq_len - 1)

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, idx: int):
        x = self.data[idx       : idx + self.seq_len]
        y = self.data[idx + 1   : idx + self.seq_len + 1]
        return x, y

In [6]:
# ---------------------------------------------------------------------------
# Training function
# ---------------------------------------------------------------------------
def train() -> None:
    cfg = TrainConfig()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # ── 1. Tokenizer ────────────────────────────────────────────────────────
    corpus = get_corpus()
    os.makedirs(cfg.save_dir, exist_ok=True)
    tok_path = os.path.join(cfg.save_dir, "tokenizer.json")

    tokenizer = BPETokenizer(num_merges=cfg.bpe_merges)
    if os.path.exists(tok_path):
        tokenizer.load(tok_path)
    else:
        tokenizer.train(corpus)
        tokenizer.save(tok_path)

    # ── 2. Encode corpus ────────────────────────────────────────────────────
    token_ids = tokenizer.encode(corpus, add_special_tokens=False)
    print(f"Corpus → {len(token_ids)} tokens | vocab size: {tokenizer.vocab_size}")

    # ── 3. Dataset & DataLoader ─────────────────────────────────────────────
    dataset    = TextDataset(token_ids, seq_len=cfg.max_seq_len)
    dataloader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True,
                            drop_last=True)
    print(f"Dataset: {len(dataset)} windows | batches/epoch: {len(dataloader)}")

    # ── 4. Model ─────────────────────────────────────────────────────────────
    model_cfg = ModelConfig(
        vocab_size  = tokenizer.vocab_size,
        d_model     = cfg.d_model,
        n_heads     = cfg.n_heads,
        n_kv_heads  = cfg.n_kv_heads,
        d_ff        = cfg.d_ff,
        max_seq_len = cfg.max_seq_len,
        dropout     = cfg.dropout,
    )
    model = MiniLLaMA(model_cfg).to(device)
    print(f"MiniLLaMA parameters: {model.num_parameters():,}")

    # ── 5. Optimizer & Scheduler ─────────────────────────────────────────────
    optimizer = AdamW(model.parameters(), lr=cfg.lr,
                      weight_decay=cfg.weight_decay)
    total_steps = cfg.epochs * len(dataloader)
    scheduler   = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)

    # ── 6. Training loop ─────────────────────────────────────────────────────
    print("\n── Training ──────────────────────────────────────────────────")
    model.train()
    for epoch in range(1, cfg.epochs + 1):
        epoch_loss = 0.0
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            _, loss = model(x, targets=y)
            loss.backward()

            # Gradient clipping (standard in LLM training)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        if epoch % cfg.log_every == 0:
            avg = epoch_loss / len(dataloader)
            lr  = scheduler.get_last_lr()[0]
            print(f"  epoch {epoch:4d}/{cfg.epochs}  loss={avg:.4f}  lr={lr:.2e}")

    # ── 7. Save checkpoint ───────────────────────────────────────────────────
    ckpt_path = os.path.join(cfg.save_dir, "minillama.pt")
    torch.save({
        "model_state_dict"  : model.state_dict(),
        "model_config"      : model_cfg.__dict__,
        "train_config"      : cfg.__dict__,
        "vocab_size"        : tokenizer.vocab_size,
    }, ckpt_path)
    print(f"\n✓ Checkpoint saved to {ckpt_path}")

In [7]:
train()

Device: cuda
[BPE] Tokenizer loaded from checkpoints/tokenizer.json  vocab_size=555
Corpus → 341 tokens | vocab size: 555
Dataset: 276 windows | batches/epoch: 34
MiniLLaMA parameters: 218,880

── Training ──────────────────────────────────────────────────
  epoch   10/200  loss=0.0368  lr=2.98e-03
  epoch   20/200  loss=0.0331  lr=2.93e-03
  epoch   30/200  loss=0.0302  lr=2.84e-03
  epoch   40/200  loss=0.0250  lr=2.71e-03
  epoch   50/200  loss=0.0235  lr=2.56e-03
  epoch   60/200  loss=0.0204  lr=2.38e-03
  epoch   70/200  loss=0.0203  lr=2.18e-03
  epoch   80/200  loss=0.0192  lr=1.97e-03
  epoch   90/200  loss=0.0185  lr=1.74e-03
  epoch  100/200  loss=0.0182  lr=1.50e-03
  epoch  110/200  loss=0.0177  lr=1.27e-03
  epoch  120/200  loss=0.0172  lr=1.04e-03
  epoch  130/200  loss=0.0163  lr=8.26e-04
  epoch  140/200  loss=0.0163  lr=6.26e-04
  epoch  150/200  loss=0.0155  lr=4.48e-04
  epoch  160/200  loss=0.0157  lr=2.96e-04
  epoch  170/200  loss=0.0150  lr=1.73e-04
  epoch  180